# IntPhys2 — Prediction-based Evaluation

## Surprisal formulation

Given a video $V$ of $N$ frames and a model handling $M < N$ frames, we form overlapping sliding windows of $M$ frames with stride $S$, giving windows starting at frames $0, S, 2S, \ldots, N{-}M$. Each window is split into a **context** of $C$ frames and a **target** of $T = M - C$ frames. The per-window surprise is:

$$\text{Surprise}_w = d\!\left(p\!\left(f(V_{w:w+C})\right),\; f'(V_{w+C:w+M})\right) \tag{1}$$

where $w$ is the window's starting frame, $d$ a distance measure, $f$ the encoder for the context, $f'$ the target encoder, and $p$ the predictor.

### V-JEPA 2

$$\text{Surprise}_w = d\!\left(p\!\left(f(V_{w:w+C})\right),\; \mathrm{LN}\!\left([f'(V_{w:w+M})]_T\right)\right)$$

- $f$ — V-JEPA 2 encoder (frozen ViT-H), applied to context patch tokens
- $f'$ — target encoder (EMA copy of $f$), applied to the **full** window; $[\cdot]_T$ selects the target patch positions $w{+}C \to w{+}M$
- $p$ — predictor (separate ViT), takes context encodings and mask tokens and predicts target-region representations
- $\mathrm{LN}$ — layer normalization over the feature dimension, applied to targets before computing $d$
- $d$ — mean L1 distance over all $N_T$ target tokens and feature dimension $D$:

$$d(\hat{z}, z) = \frac{1}{N_T \cdot D}\sum_{n=1}^{N_T}\sum_{d=1}^{D} |\hat{z}_{n,d} - z_{n,d}|$$

Surprise is measured in **representation space** (not pixel space). The EMA target encoder provides stable targets decoupled from the online encoder weights.

### VideoMAEv2

For VideoMAEv2, $f$ is the encoder, $f'$ is the **identity** (patch-normalized pixel values), and $p$ is the decoder — so surprise is measured directly in pixel space.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

# figures and computations


In [22]:
# ---------------------------------------------------------------------------
# Configuration — edit these before running
# ---------------------------------------------------------------------------

MODEL_NAME  = "vjepa2"          
# label used in plot titles and filenames
PTH_FOLDER  = "/data/kxzheng/models/checkpoint/vjepa-2-h/intphys_v2/intphys2-main-default"                
# path to folder containing losses_*.pth files
METADATA_CSV = "/data/kxzheng/data/IntPhys2/Debug/metadata.csv"               
# path to metadata.csv for this split
DATASET     = "intphys2-main" # intphys2-debug | intphys2-main | intphys2-heldout

# Scene to plot in the surprise-over-time section
SCENE_IDX   = None              # set to an integer SceneIndex, or None to use the first scene
CTX_IDX     = 0                 # index into context_lengths list saved in the .pth

# Where to save figures (set to None to skip saving)
SAVE_DIR    = "./figures"

In [ ]:
# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_pth_files(pth_folder, dataset):
    if dataset == "intphys2-main":
        files = sorted(glob.glob(os.path.join(pth_folder, "losses_*rank*.pth")))
    else:
        files = sorted(glob.glob(os.path.join(pth_folder, "losses_*.pth")))
        files = [f for f in files if "rank" not in os.path.basename(f)]
    if not files:
        raise FileNotFoundError(f"No .pth files found in {pth_folder}")
    all_losses, all_names, meta = [], [], None
    for f in files:
        data = torch.load(f, map_location="cpu", weights_only=False)
        if meta is None:
            meta = {k: v for k, v in data.items() if k not in ("losses", "names")}
        all_losses.append(data["losses"])
        all_names.extend(list(data["names"]))
    max_t = max(l.shape[2] for l in all_losses)
    padded = [torch.nn.functional.pad(l, (0, max_t - l.shape[2])) for l in all_losses]
    return torch.cat(padded, dim=0), all_names, meta   # [N, C, T]


def load_metadata(metadata_csv):
    df_gt_labels = pd.read_csv(metadata_csv)
    df_gt_labels["_basename"] = df_gt_labels["file_name"].apply(lambda x: os.path.basename(str(x)))
    df_gt_labels = df_gt_labels.rename(columns={'name': 'filename'})
    #predicted_labels.filename = predicted_labels.filename.astype(str)
    df_gt_labels.filename = df_gt_labels.filename.astype(str)
    return df_gt_labels

def label_target(row):
    if "Impossible" in row["type"]:
        return 0
    elif "Possible" in row["type"]:
        return 1
    else:
        print("Parsing Error!")

def aggregate_surprise(losses):
    """Mean L1 loss per video, ignoring zero-padded windows → [N_videos]."""
    mask = (losses > 0).float()
    return ((losses * mask).sum(dim=(1, 2)) / mask.sum(dim=(1, 2)).clamp(min=1)).numpy()

def build_results_df(losses, names, metadata_df):
    scores = aggregate_surprise(losses)
    name_to_idx = {n: i for i, n in enumerate(names)}
    df = metadata_df.copy()
    df["video_idx"] = df["_basename"].map(name_to_idx)
    df["surprise_score"] = df["video_idx"].apply(
        lambda idx: scores[int(idx)] if pd.notna(idx) else np.nan
    )
    df = df.dropna(subset=["surprise_score"])
    df["video_idx"] = df["video_idx"].astype(int)
    df["target"] = df.apply(label_target, axis=1)
    df["target"] = df["target"].astype(int)
    return df


def compute_scene_accuracy(results_df):
    rows = []
    for scene_idx, grp in results_df.groupby("SceneIndex"):
        possible   = grp[grp["target"] == 1]["surprise_score"]
        impossible = grp[grp["target"] == 0]["surprise_score"]
        # print('p', possible)
        # print('imp', impossible)
        if possible.empty or impossible.empty:
            continue
        rows.append({
            "SceneIndex": scene_idx,
            "correct":    int(impossible.mean() > possible.mean()),
            "condition":  grp["condition"].iloc[0],
            "env":        grp["env"].iloc[0],
            "Difficulty": grp["Difficulty"].iloc[0],
            "game_name":  grp["game_name"].iloc[0],
            "Cam":        grp["Camera"].iloc[0],
        })
    return pd.DataFrame(rows)


def savefig(fig, name):
    if SAVE_DIR is None:
        return
    os.makedirs(SAVE_DIR, exist_ok=True)
    path = os.path.join(SAVE_DIR, f"{MODEL_NAME}_{name}.png")
    fig.savefig(path, bbox_inches="tight")
    print(f"Saved: {path}")

In [24]:
# ---------------------------------------------------------------------------
# Load data
# ---------------------------------------------------------------------------

losses, names, meta = load_pth_files(PTH_FOLDER, DATASET)
metadata_df         = load_metadata(METADATA_CSV)
results_df          = build_results_df(losses, names, metadata_df)
scene_acc           = compute_scene_accuracy(results_df)
ctx_lengths         = meta.get("context_lengths", list(range(losses.shape[1])))

p 0    0.680961
3    0.681719
Name: surprise_score, dtype: float64
imp 1    0.682629
2    0.682517
Name: surprise_score, dtype: float64
p 12    0.688523
15    0.687287
Name: surprise_score, dtype: float64
imp 13    0.684781
14    0.691482
Name: surprise_score, dtype: float64


In [11]:
results_df.head()

,SceneIndex,filename,file_name,game_name,condition,env,type,occluder,Difficulty,Camera,_basename,video_idx,surprise_score,target
0,0,083861b17a8b3f4bef7e3aaad0b960318a5ea19ca98258...,Videos/083861b17a8b3f4bef7e3aaad0b960318a5ea19...,FixedJumpSolidity,solidity,BasicLevel_0,1_Possible,NaN,Easy,Fixed,083861b17a8b3f4bef7e3aaad0b960318a5ea19ca98258...,15,0.682091,1
1,0,56c044c79b791ff1d2eae592c3af1914772d79a23c9603...,Videos/56c044c79b791ff1d2eae592c3af1914772d79a...,FixedJumpSolidity,solidity,BasicLevel_0,1_Impossible,NaN,Easy,Fixed,56c044c79b791ff1d2eae592c3af1914772d79a23c9603...,6,0.683500,0
2,0,4b2779c2649b64094af506b8783525387c2f7cd52dd8f5...,Videos/4b2779c2649b64094af506b8783525387c2f7cd...,FixedJumpSolidity,solidity,BasicLevel_0,2_Impossible,NaN,Easy,Fixed,4b2779c2649b64094af506b8783525387c2f7cd52dd8f5...,20,0.684354,0
3,0,66a6de26327fe7661ac4fec04da96d0710d2177afad8ba...,Videos/66a6de26327fe7661ac4fec04da96d0710d2177...,FixedJumpSolidity,solidity,BasicLevel_0,2_Possible,NaN,Easy,Fixed,66a6de26327fe7661ac4fec04da96d0710d2177afad8ba...,51,0.683731,1
4,0_1,1f96abb75a7df3f52384aefd21bef910687850138b7353...,Videos/1f96abb75a7df3f52384aefd21bef9106878501...,FixedJumpSolidity,solidity,BasicLevel_0,1_Possible,NaN,Easy,Fixed,1f96abb75a7df3f52384aefd21bef910687850138b7353...,32,0.683979,1


In [19]:
def compute_scene_accuracy(results_df):
    rows = []
    for scene_idx, grp in results_df.groupby("SceneIndex"):
        possible   = grp[grp["target"] == 1]["surprise_score"]
        impossible = grp[grp["target"] == 0]["surprise_score"]
        print('p', possible)
        print('imp', impossible)
        if possible.empty or impossible.empty:
            continue
        rows.append({
            "SceneIndex": scene_idx,
            "correct":    int(impossible.mean() > possible.mean()),
            "condition":  grp["condition"].iloc[0],
            "env":        grp["env"].iloc[0],
            "Difficulty": grp["Difficulty"].iloc[0],
            "game_name":  grp["game_name"].iloc[0],
            "Cam":        grp["Camera"].iloc[0],
        })
    return pd.DataFrame(rows)

compute_scene_accuracy(results_df)

p 0    0.682091
3    0.683731
Name: surprise_score, dtype: float32
imp 1    0.683500
2    0.684354
Name: surprise_score, dtype: float32
p 4    0.683979
7    0.682139
Name: surprise_score, dtype: float32
imp 5    0.685101
6    0.682728
Name: surprise_score, dtype: float32
p 8     0.684489
11    0.684177
Name: surprise_score, dtype: float32
imp 9     0.683622
10    0.683834
Name: surprise_score, dtype: float32
p 12    0.690277
15    0.688534
Name: surprise_score, dtype: float32
imp 13    0.687032
14    0.691882
Name: surprise_score, dtype: float32
p 16    0.689241
19    0.687456
Name: surprise_score, dtype: float32
imp 17    0.687796
18    0.690662
Name: surprise_score, dtype: float32
p 20    0.689554
23    0.686315
Name: surprise_score, dtype: float32
imp 21    0.687370
22    0.690601
Name: surprise_score, dtype: float32
p 24    0.679830
27    0.679847
Name: surprise_score, dtype: float32
imp 25    0.678706
26    0.680523
Name: surprise_score, dtype: float32
p 28    0.678246
31    0.680

,SceneIndex,correct,condition,env,Difficulty,game_name,Cam
0,0,1,solidity,BasicLevel_0,Easy,FixedJumpSolidity,Fixed
1,0_1,1,solidity,BasicLevel_0,Easy,FixedJumpSolidity,Fixed
2,0_2,0,solidity,BasicLevel_0,Easy,FixedJumpSolidity,Fixed
3,1,1,solidity,BasicLevel_0,Easy,SolidityFallingFlat,Fixed
4,1_1,1,solidity,BasicLevel_0,Easy,SolidityFallingFlat,Fixed
5,1_2,1,solidity,BasicLevel_0,Easy,SolidityFallingFlat,Fixed
6,2,0,solidity,BasicLevel_0,Easy,FixedMarryPoppins,Fixed
7,2_1,1,solidity,BasicLevel_0,Easy,FixedMarryPoppins,Fixed
8,2_2,1,solidity,BasicLevel_0,Easy,FixedMarryPoppins,Fixed
9,3,1,permanence,BasicLevel_0,Easy,HotAirBallonTwoDist,Fixed


In [21]:
overall = scene_acc["correct"].mean() * 100
print(f"Model     : {MODEL_NAME}")
print(f"Dataset   : {DATASET}")
print(f"Videos    : {losses.shape[0]} in .pth | {len(results_df)} matched in metadata")
print(f"Scenes    : {len(scene_acc)}")
print(f"Overall   : {overall:.1f}%")
print(f"Ctx lens  : {ctx_lengths}")
print(f"Loss shape: {list(losses.shape)}  [N_videos, N_ctx, N_time_steps]")

Model     : vjepa2
Dataset   : intphys2-debug
Videos    : 60 in .pth | 60 matched in metadata
Scenes    : 15
Overall   : 80.0%
Ctx lens  : [12, 18, 24, 30, 36, 42]
Loss shape: [60, 6, 29]  [N_videos, N_ctx, N_time_steps]


In [ ]:
# ---------------------------------------------------------------------------
# Accuracy — overall
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(4, 3.5))
bar = ax.bar([MODEL_NAME], [overall], color="steelblue", edgecolor="white")
ax.axhline(50, color="tomato", linestyle="--", linewidth=1.3, label="Chance (50%)")
ax.set_ylim(0, 105)
ax.set_ylabel("Accuracy (%)")
ax.set_title(f"Overall accuracy — {MODEL_NAME} ({DATASET.split('-')[-1]})")
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
ax.text(bar[0].get_x() + bar[0].get_width()/2, overall + 1.5,
        f"{overall:.1f}%", ha="center", va="bottom", fontsize=11)
plt.tight_layout()
savefig(fig, "accuracy_overall")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Accuracy — grouped bar charts
# ---------------------------------------------------------------------------

GROUP_COLS = ["condition", "Difficulty", "env", "game_name", "Cam"]

for group in GROUP_COLS:
    agg = (
        scene_acc.groupby(group)["correct"]
        .agg(["mean", "count"])
        .sort_values("mean", ascending=False)
        .reset_index()
    )
    fig, ax = plt.subplots(figsize=(max(5, len(agg) * 1.2), 4))
    xs   = range(len(agg))
    bars = ax.bar(xs, agg["mean"] * 100, color="steelblue", edgecolor="white")
    ax.axhline(50, color="tomato", linestyle="--", linewidth=1.3, label="Chance (50%)")
    ax.set_xticks(list(xs))
    ax.set_xticklabels(
        [f"{row[group]}\nn={row['count']}" for _, row in agg.iterrows()],
        rotation=30, ha="right", fontsize=9,
    )
    ax.set_ylim(0, 115)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title(f"{MODEL_NAME} — accuracy by {group}")
    ax.legend(fontsize=9)
    ax.grid(axis="y", alpha=0.3)
    for bar, (_, row) in zip(bars, agg.iterrows()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
                f"{row['mean']*100:.1f}%", ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    savefig(fig, f"accuracy_{group}")
    plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Surprise over time — set SCENE_IDX and CTX_IDX in the config cell
# ---------------------------------------------------------------------------

scene_idx = SCENE_IDX if SCENE_IDX is not None else int(results_df["SceneIndex"].iloc[0])
ctx_idx   = CTX_IDX
ctx_len   = ctx_lengths[ctx_idx]

scene_df  = results_df[results_df["SceneIndex"] == scene_idx]
scene_row = scene_df[["condition", "Difficulty", "game_name"]].iloc[0]
palette   = {"Possible": "#3a86ff", "Impossible": "#e63946"}

fig, ax = plt.subplots(figsize=(12, 4))

for _, row in scene_df.iterrows():
    vidx  = int(row["video_idx"])
    trace = losses[vidx, ctx_idx, :].numpy()
    nz    = np.where(trace > 0)[0]
    if len(nz) == 0:
        continue
    trace = trace[: nz[-1] + 1]
    color = palette.get(row["type"], "gray")
    label = f"{row['type']} — {os.path.basename(str(row['file_name']))}"
    ax.plot(trace, color=color, linewidth=2, alpha=0.9, label=label)
    ax.fill_between(range(len(trace)), trace, alpha=0.08, color=color)

ax.set_title(
    f"{MODEL_NAME}  |  Scene {scene_idx}  "
    f"({scene_row['condition']}, {scene_row['Difficulty']}, {scene_row['game_name']})  "
    f"|  context = {ctx_len}"
)
ax.set_xlabel("Time step (window index)")
ax.set_ylabel("L1 surprise")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout()
savefig(fig, f"surprise_scene{scene_idx}_ctx{ctx_len}")
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Surprise over time — all context lengths for the selected scene
# ---------------------------------------------------------------------------

n_ctx = len(ctx_lengths)
ncols = min(n_ctx, 3)
nrows = (n_ctx + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3.5 * nrows), squeeze=False)

for i, (c_len, ax) in enumerate(zip(ctx_lengths, axes.flatten())):
    for _, row in scene_df.iterrows():
        vidx  = int(row["video_idx"])
        trace = losses[vidx, i, :].numpy()
        nz    = np.where(trace > 0)[0]
        if len(nz) == 0:
            continue
        trace = trace[: nz[-1] + 1]
        color = palette.get(row["type"], "gray")
        ax.plot(trace, color=color, linewidth=1.8, alpha=0.9,
                label=row["type"] if i == 0 else "_")
        ax.fill_between(range(len(trace)), trace, alpha=0.08, color=color)
    ax.set_title(f"context = {c_len}", fontsize=9)
    ax.set_xlabel("Time step")
    ax.set_ylabel("L1 surprise")
    ax.grid(alpha=0.3)

# Hide unused axes
for ax in axes.flatten()[n_ctx:]:
    ax.set_visible(False)

# Shared legend from first subplot
handles, labels = axes.flatten()[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper right", fontsize=9)
fig.suptitle(
    f"{MODEL_NAME}  |  Scene {scene_idx}  "
    f"({scene_row['condition']}, {scene_row['Difficulty']})  — all context lengths",
    fontsize=11, y=1.01,
)
plt.tight_layout()
savefig(fig, f"surprise_scene{scene_idx}_all_ctx")
plt.show()